# United States Presidential Debates | Official Corpus Creation

In [28]:
import os
import re
import json
import requests
from bs4 import BeautifulSoup
from collections import OrderedDict

In [79]:
debate_dict = OrderedDict([
    ("2020", {
        "info": "https://www.debates.org/debate-history/2020-debates/",
        "transcripts": [
            ("09-29", "https://www.debates.org/voter-education/debate-transcripts/september-29-2020-debate-transcript/"),
            ("10-07", "https://www.debates.org/voter-education/debate-transcripts/vice-presidential-debate-at-the-university-of-utah-in-salt-lake-city-utah/"),
            ("10-22", "https://www.debates.org/voter-education/debate-transcripts/october-22-2020-debate-transcript/")
        ]
    }),
    ("2016", {
        "info": "https://www.debates.org/debate-history/2016-debates/",
        "transcripts": [
            ("09-26", "https://www.debates.org/voter-education/debate-transcripts/september-26-2016-debate-transcript/"),
            ("10-04", "https://www.debates.org/voter-education/debate-transcripts/october-4-2016-debate-transcript/"),
            ("10-09", "https://www.debates.org/voter-education/debate-transcripts/october-9-2016-debate-transcript/"),
            ("10-19", "https://www.debates.org/voter-education/debate-transcripts/october-19-2016-debate-transcript/")
        ]
    }),
    ("2012", {
        "info": "https://www.debates.org/debate-history/2012-debates/",
        "transcripts": [
            ("10-03", "https://www.debates.org/voter-education/debate-transcripts/october-3-2012-debate-transcript/"),
            ("10-11", "https://www.debates.org/voter-education/debate-transcripts/october-11-2012-the-biden-romney-vice-presidential-debate/"),
            ("10-16", "https://www.debates.org/voter-education/debate-transcripts/october-16-2012-the-second-obama-romney-presidential-debate/"),
            ("10-22", "https://www.debates.org/voter-education/debate-transcripts/october-22-2012-the-third-obama-romney-presidential-debate/")
        ]
    }),
    ("2008", {
        "info": "https://www.debates.org/debate-history/2008-debates/",
        "transcripts": [
            ("09-26", "https://www.debates.org/voter-education/debate-transcripts/2008-debate-transcript/"),
            ("10-07", "https://www.debates.org/voter-education/debate-transcripts/october-7-2008-debate-transcrip/"),
            ("10-15", "https://www.debates.org/voter-education/debate-transcripts/october-15-2008-debate-transcript/")
        ]
    }),
    ("2004", {
        "info": "https://www.debates.org/debate-history/2004-debates/",
        "transcripts": [
            ("09-30", "https://www.debates.org/voter-education/debate-transcripts/september-30-2004-debate-transcript/"),
            ("10-05", "https://www.debates.org/voter-education/debate-transcripts/october-5-2004-transcript/"),
            ("10-08", "https://www.debates.org/voter-education/debate-transcripts/october-8-2004-debate-transcript/"),
            ("10-13", "https://www.debates.org/voter-education/debate-transcripts/october-13-2004-debate-transcript/")
        ]
    }),
    ("2000", {
        "info": "https://www.debates.org/debate-history/2000-debates/",
        "transcripts": [
            ("10-03", "https://www.debates.org/voter-education/debate-transcripts/october-3-2000-transcript/"),
            ("10-05", "https://www.debates.org/voter-education/debate-transcripts/october-5-2000-debate-transcript/"),
            ("10-11", "https://www.debates.org/voter-education/debate-transcripts/october-11-2000-debate-transcript/"),
            ("10-17", "https://www.debates.org/voter-education/debate-transcripts/october-17-2000-debate-transcript/")
        ]
    }),
    ("1996", {
        "info": "https://www.debates.org/debate-history/1996-debates/",
        "transcripts": [
            ("10-06", "https://www.debates.org/voter-education/debate-transcripts/october-6-1996-debate-transcript/"),
            ("10-09", "https://www.debates.org/voter-education/debate-transcripts/october-9-1996-debate-transcript/"),
            ("10-16", "https://www.debates.org/voter-education/debate-transcripts/october-16-1996-debate-transcript/")
        ]
    }),
    ("1992", {
        "info": "https://www.debates.org/debate-history/1992-debates/",
        "transcripts": [
            ("10-11", "https://www.debates.org/voter-education/debate-transcripts/october-11-1992-first-half-debate-transcript/"),
            ("10-11", "https://www.debates.org/voter-education/debate-transcripts/october-11-1992-second-half-debate-transcript/"),
            ("10-13", "https://www.debates.org/voter-education/debate-transcripts/october-13-1992-debate-transcript/"),
            ("10-19", "https://www.debates.org/voter-education/debate-transcripts/october-19-1992-debate-transcript/")
        ]
    }),
    ("1988", {
        "info": "https://www.debates.org/debate-history/1988-debates/",
        "transcripts": [
            ("09-25", "https://www.debates.org/voter-education/debate-transcripts/september-25-1988-debate-transcript/"),
            ("10-05", "https://www.debates.org/voter-education/debate-transcripts/october-5-1988-debate-transcripts/"),
            ("10-13", "https://www.debates.org/voter-education/debate-transcripts/october-13-1988-debate-transcript/")
        ]
    }),
    ("1984", {
        "info": "https://www.debates.org/debate-history/1984-debates/",
        "transcripts": [
            ("10-07", "https://www.debates.org/voter-education/debate-transcripts/october-7-1984-debate-transcript/"),
            ("10-11", "https://www.debates.org/voter-education/debate-transcripts/october-11-1984-debate-transcript/"),
            ("10-21", "https://www.debates.org/voter-education/debate-transcripts/october-21-1984-debate-transcript/")
        ]
    }),
    ("1980", {
        "info": "https://www.debates.org/debate-history/1980-debates/",
        "transcripts": [
            ("09-21", "https://www.debates.org/voter-education/debate-transcripts/september-21-1980-debate-transcript/"),
            ("10-28", "https://www.debates.org/voter-education/debate-transcripts/october-28-1980-debate-transcript/")
        ]
    }),
    ("1976", {
        "info": "https://www.debates.org/debate-history/1976-debates/",
        "transcripts": [
            ("09-23", "https://www.debates.org/voter-education/debate-transcripts/september-23-1976-debate-transcript/"),
            ("10-06", "https://www.debates.org/voter-education/debate-transcripts/october-6-1976-debate-transcript/"),
            ("10-15", "Transcript not available."),
            ("10-22", "https://www.debates.org/voter-education/debate-transcripts/october-22-1976-debate-transcript/")
        ]
    }),
    ("1960", {
        "info": "https://www.debates.org/debate-history/1960-debates/",
        "transcripts": [
            ("09-26", "https://www.debates.org/voter-education/debate-transcripts/september-26-1960-debate-transcript/"),
            ("10-07", "https://www.debates.org/voter-education/debate-transcripts/october-7-1960-debate-transcript/"),
            ("10-13", "https://www.debates.org/voter-education/debate-transcripts/october-13-1960-debate-transcript/"),
            ("10-21", "https://www.debates.org/voter-education/debate-transcripts/october-21-1960-debate-transcript/")
        ]
    }),
    ("1956", {
        "info": "https://www.debates.org/debate-history/1956-debate/",
        "transcripts": [
            ("05-21", "Transcript not available.")
        ]
    }),
    ("1948", {
        "info": "https://www.debates.org/debate-history/1948-debate/",
        "transcripts": [
            ("05-17", "Transcript not available.")
        ]
    }),
    ("1858", {
        "info": "https://www.debates.org/debate-history/1858-debates/",
        "transcripts": [
            ("08-21", "Transcript not available."),
            ("08-27", "Transcript not available."),
            ("09-15", "Transcript not available."),
            ("09-18", "Transcript not available."),
            ("10-07", "Transcript not available."),
            ("10-13", "Transcript not available."),
            ("10-15", "Transcript not available.")
        ]
    })
])


In [74]:
import os
import requests
from bs4 import BeautifulSoup

# The CSS selector to extract the content
CONTENT_SELECTOR = "#content-sm"

# Base directory to save scraped content
BASE_DIR = "all_debates"

# Log file name
LOG_FILE = "log.txt"

def extract_content_from_url(url, selector):
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')
    selected_content = soup.select_one(selector)
    return selected_content.get_text() if selected_content else None

def write_content_to_file(directory, filename, content):
    os.makedirs(directory, exist_ok=True)
    with open(os.path.join(directory, filename), 'w', encoding='utf8') as file:
        file.write(content)

def log_message(message):
    with open(LOG_FILE, 'a', encoding='utf8') as log_file:
        log_file.write(message + "\n")

def scrape_all_content_and_save(debate_dict):
    for year, data in debate_dict.items():
        raw_directory = os.path.join(BASE_DIR, year, "raw")

        # Scrape the info page
        info_url = data["info"]
        info_content = extract_content_from_url(info_url, CONTENT_SELECTOR)
        if info_content:
            log_message(f"Successfully scraped {info_url}")
            write_content_to_file(raw_directory, "raw-info.txt", info_content)
            log_message(f"Saved info content as {os.path.join(raw_directory, 'raw-info.txt')}")
        else:
            log_message(f"Error scraping {info_url}")
            continue  # Skip to the next year if scraping fails

        # Scrape the debate transcripts
        for i, (date, transcript_url) in enumerate(data["transcripts"], start=1):
            if transcript_url != "Transcript not available.":
                transcript_content = extract_content_from_url(transcript_url, CONTENT_SELECTOR)
                if transcript_content:
                    log_message(f"Successfully scraped {transcript_url}")
                    write_content_to_file(raw_directory, f"{year}-{date}-debate-raw.txt", transcript_content)
                    log_message(f"Saved transcript content as {os.path.join(raw_directory, f'{year}-{date}-debate-raw.txt')}")
                else:
                    log_message(f"Error scraping {transcript_url}")


In [78]:
scrape_all_content_and_save(debate_dict)


In [81]:
import os
import re

def log_message(log_file_path, message):
    with open(log_file_path, 'a', encoding='utf8') as log_file:
        log_file.write(message + "\n")

def process_transcripts(log_file_path, base_dir):
    for root, dirs, files in os.walk(base_dir):
        for file in files:
            if file.startswith("clean-") and file.endswith(".txt"):
                file_path = os.path.join(root, file)
                with open(file_path, 'r', encoding='utf8') as content_file:
                    content = content_file.read()

                # Ensure no empty lines
                lines = [line for line in content.split("\n") if line.strip() != ""]

                # Ensure every line starts with speaker name
                lines = [f"{re.split(':', line, 1)[0].upper()}:{re.split(':', line, 1)[1]}" if ":" in line else line for line in lines]

                # Remove any '(...)' or '[...]' from start of line and add to previous line
                for i in range(1, len(lines)):
                    if lines[i].startswith(("(", "[")):
                        lines[i-1] += " " + lines[i]
                        lines[i] = ""

                # Combine lines into single string
                content = "\n".join(line for line in lines if line)

                # Write processed content back to file
                with open(file_path, 'w', encoding='utf8') as content_file:
                    content_file.write(content)

                # Write log
                log_message(log_file_path, f"Processed and wrote cleaned content to {file_path}")

# Create a directory for logs if it doesn't exist
os.makedirs("logs", exist_ok=True)
log_file_path = "logs/clean_transcript_log.txt"

base_dir = "all_debates"
process_transcripts(log_file_path, base_dir)

In [1]:
import nltk
nltk.download('punkt')

from nltk.tokenize import sent_tokenize, word_tokenize

def count_sentences_and_words(*filenames):
    for filename in filenames:
        with open(filename, 'r') as file:
            text = file.read()
            sentences = sent_tokenize(text)
            words = word_tokenize(text)
            print(f"{filename} - Sentences: {len(sentences)}, Words: {len(words)}")



[nltk_data] Downloading package punkt to
[nltk_data]     /Users/pulpdesign/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [10]:
x = "clean-2020-09-29-debate copy.txt"
y = "clean-2020-09-29-debate.txt"

In [12]:
count_sentences_and_words(x, y)

clean-2020-09-29-debate copy.txt - Sentences: 1913, Words: 25925
clean-2020-09-29-debate.txt - Sentences: 1913, Words: 25925


In [13]:
import re
import os

# Log file path
LOG_FILE_PATH = "clean_transcript_log.txt"

def log_message(message):
    with open(LOG_FILE_PATH, 'a', encoding='utf8') as log_file:
        log_file.write(message + "\n")

def clean_transcript(transcript_file):
    with open(transcript_file, 'r', encoding='utf-8') as file:
        lines = file.readlines()

    new_lines = []
    current_speaker = ""
    for line in lines:
        # If line is empty or contains only whitespace, skip it
        if not line.strip():
            continue

        # If the line starts with a speaker name and colon
        match = re.match(r"^([\w\s,]+):", line)
        if match:
            original_speaker = match.group(1)
            current_speaker = original_speaker.split(",")[0].split()[-1] + ':'  # Keep only the last word before the first comma
            new_line = line.replace(original_speaker+":", current_speaker)
            new_lines.append(new_line)
            
            if original_speaker != current_speaker.strip(':'):
                log_message(f"File: {transcript_file}, Replaced speaker '{original_speaker}' with '{current_speaker.strip(':')}', Line: '{line.strip()}' -> '{new_line.strip()}'")

        else:
            # If line doesn't start with speaker name and colon, append it with the current speaker's name
            new_line = current_speaker + " " + line
            new_lines.append(new_line)
            log_message(f"File: {transcript_file}, Prepended speaker '{current_speaker.strip(':')}' to line: '{line.strip()}' -> '{new_line.strip()}'")

    # Save cleaned transcript
    with open(transcript_file, 'w', encoding='utf-8') as file:
        file.writelines(new_lines)


def process_transcripts(base_directory):
    for root, dirs, files in os.walk(base_directory):
        for file in files:
            if file.startswith("clean"):
                clean_transcript(os.path.join(root, file))


process_transcripts("all_debates")